# Training and fine tuning

z

In [ ]:
import optuna
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
categorical_features = ['resto_name',]

In [ ]:
tune_split_idx = int(len(X_train) * 0.8)

X_tune_train, y_tune_train = X_train.iloc[:tune_split_idx], y_train.iloc[:tune_split_idx]
X_tune_valid, y_tune_valid = X_train.iloc[tune_split_idx:], y_train.iloc[tune_split_idx:]

In [ ]:
def objective(trial):
    # Define the hyperparameter search space
    param = {
        'iterations': 1000, # Keep high, rely on early stopping
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'random_strength': trial.suggest_float('random_strength', 1e-9, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        
        'loss_function': 'MAE',
        'eval_metric': 'MAE',
        'random_seed': 42,
        'cat_features': categorical_features,
        'verbose': False
    }

    # Initialize model
    model = CatBoostRegressor(**param)

    # Fit the model with early stopping
    model.fit(
        X_tune_train, y_tune_train,
        eval_set=[(X_tune_valid, y_tune_valid)],
        early_stopping_rounds=50, # Stops if MAE doesn't improve for 50 trees
        use_best_model=True
    )

    # Predict on the validation set
    preds = model.predict(X_tune_valid)
    
    # Calculate MAE
    mae = mean_absolute_error(y_tune_valid, preds)
    return mae

In [ ]:
print("Starting hyperparameter tuning...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30) # 30 trials is a good starting point

print("\n--- Tuning Complete ---")
print(f"Best MAE: {study.best_value}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")